In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/processed', exist_ok=True)

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
synthetic = pd.read_csv('../data/synthetic/synthetic_data.csv')
print(f"✅ Synthetic data loaded: {synthetic.shape}")
print(f"   Columns: {list(synthetic.columns)}")

✅ Synthetic data loaded: (50000, 22)
   Columns: ['age', 'gender', 'height', 'weight', 'bmi', 'avg_systolic', 'avg_diastolic', 'bp_variability', 'avg_glucose', 'sugar_variability', 'avg_pulse', 'adherence_rate', 'missed_doses', 'has_chronic_condition', 'allergies_count', 'physical_activity_score', 'diet_quality_score', 'stress_score', 'sleep_score', 'diabetes_risk', 'hypertension_risk', 'health_score']


In [4]:
pima = pd.read_csv('../data/raw/pima_diabetes.csv')
print(pima.head(5).to_string())
print(f"\nColumns: {list(pima.columns)}")
print(f"Shape: {pima.shape}")

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  DiabetesPedigreeFunction  Age  Outcome
0            6      148             72             35        0  33.6                     0.627   50        1
1            1       85             66             29        0  26.6                     0.351   31        0
2            8      183             64              0        0  23.3                     0.672   32        1
3            1       89             66             23       94  28.1                     0.167   21        0
4            0      137             40             35      168  43.1                     2.288   33        1

Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Shape: (768, 9)


In [5]:
# Drop irrelevant columns
pima = pima.drop(columns=['Pregnancies', 'SkinThickness', 'Insulin', 'DiabetesPedigreeFunction'])

# Replace 0s with NaN where 0 is impossible
impossible_zero_cols = ['Glucose', 'BloodPressure', 'BMI']
pima[impossible_zero_cols] = pima[impossible_zero_cols].replace(0, np.nan)

# Fill missing with median
pima = pima.fillna(pima.median(numeric_only=True))

# Rename to our standard column names
pima = pima.rename(columns={
    'Glucose': 'avg_glucose',
    'BloodPressure': 'avg_diastolic',
    'BMI': 'bmi',
    'Age': 'age',
    'Outcome': 'diabetes_label'
})

# Add missing columns with realistic defaults
pima['avg_systolic'] = (pima['avg_diastolic'] * 1.4 + np.random.normal(0, 5, len(pima))).clip(85, 200).round(1)
pima['bp_variability'] = np.random.normal(8, 3, len(pima)).clip(0, 25).round(1)
pima['sugar_variability'] = np.random.normal(12, 4, len(pima)).clip(0, 40).round(1)
pima['avg_pulse'] = np.random.normal(75, 10, len(pima)).clip(50, 110).round(1)
pima['adherence_rate'] = np.random.normal(0.80, 0.12, len(pima)).clip(0.1, 1.0).round(2)
pima['missed_doses'] = np.round((1 - pima['adherence_rate']) * 21).astype(int).clip(0, 15)
pima['has_chronic_condition'] = (pima['diabetes_label'] == 1).astype(int)
pima['allergies_count'] = np.random.choice([0,1,2,3], len(pima), p=[0.5,0.25,0.15,0.10])
pima['physical_activity_score'] = np.random.normal(5, 2, len(pima)).clip(0, 10).round(1)
pima['diet_quality_score'] = np.random.normal(5.5, 2, len(pima)).clip(0, 10).round(1)
pima['stress_score'] = np.random.normal(5, 2, len(pima)).clip(0, 10).round(1)
pima['sleep_score'] = np.random.normal(6, 1.5, len(pima)).clip(0, 10).round(1)

# Convert label to risk levels based on glucose
def assign_diabetes_risk(row):
    if row['avg_glucose'] > 140:
        return 'high'
    elif row['avg_glucose'] > 100:
        return 'medium'
    else:
        return 'low'

pima['diabetes_risk'] = pima.apply(assign_diabetes_risk, axis=1)

print(f"✅ PIMA cleaned: {pima.shape}")
print(f"\nDiabetes Risk Distribution:")
print(pima['diabetes_risk'].value_counts())
print(pima[['age', 'bmi', 'avg_glucose', 'adherence_rate', 'diabetes_risk']].head(3))

✅ PIMA cleaned: (768, 18)

Diabetes Risk Distribution:
diabetes_risk
medium    367
low       209
high      192
Name: count, dtype: int64
   age   bmi  avg_glucose  adherence_rate diabetes_risk
0   50  33.6        148.0            0.44          high
1   31  26.6         85.0            0.76           low
2   32  23.3        183.0            0.65          high


In [6]:
cdc = pd.read_csv('../data/raw/cdc_diabetes.csv')
print(cdc.head(3).to_string())
print(f"\nColumns: {list(cdc.columns)}")
print(f"Shape: {cdc.shape}")

   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  AnyHealthcare  NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex  Age  Education  Income
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0                   0.0           0.0     0.0      1.0                0.0            1.0          0.0      5.0      18.0      15.0       1.0  0.0  9.0        4.0     3.0
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0                   0.0           1.0     0.0      0.0                0.0            0.0          1.0      3.0       0.0       0.0       0.0  0.0  7.0        6.0     1.0
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0                   0.0           0.0     1.0      0.0                0.0            1.0          1.0      5.0      30.0      30.0       1.0  0.0  9.0        4.0     8.0

Columns: ['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck'

In [7]:
# Rename columns
cdc = cdc.rename(columns={
    'Diabetes_012': 'diabetes_label',
    'BMI': 'bmi',
    'Age': 'age_group',
    'PhysActivity': 'physical_activity_raw',
    'MentHlth': 'mental_health_days',
    'PhysHlth': 'physical_health_days',
    'GenHlth': 'general_health'
})

# Convert age groups to actual ages
# CDC: 1=18-24, 2=25-29, 3=30-34... 13=80+
age_mapping = {
    1:21, 2:27, 3:32, 4:37, 5:42,
    6:47, 7:52, 8:57, 9:62, 10:67,
    11:72, 12:77, 13:82
}
cdc['age'] = cdc['age_group'].map(age_mapping).fillna(45)

# Derive glucose from diabetes label and HighBP
cdc['avg_glucose'] = (
    80 +
    (cdc['diabetes_label'] * 40) +
    (cdc['HighBP'] * 15) +
    (cdc['bmi'] * 0.8) +
    np.random.normal(0, 12, len(cdc))
).clip(60, 350).round(1)

# Derive BP from HighBP flag
cdc['avg_systolic'] = (
    110 +
    (cdc['HighBP'] * 28) +
    (cdc['bmi'] * 0.3) +
    np.random.normal(0, 8, len(cdc))
).clip(85, 200).round(1)

cdc['avg_diastolic'] = (
    cdc['avg_systolic'] * 0.65 +
    np.random.normal(0, 5, len(cdc))
).clip(55, 120).round(1)

cdc['bp_variability'] = np.random.normal(8, 3, len(cdc)).clip(0, 25).round(1)
cdc['sugar_variability'] = (
    5 + (cdc['diabetes_label'] * 8) +
    np.random.normal(0, 3, len(cdc))
).clip(0, 40).round(1)

cdc['avg_pulse'] = np.random.normal(75, 10, len(cdc)).clip(50, 110).round(1)

# Physical activity score from binary flag
cdc['physical_activity_score'] = (
    cdc['physical_activity_raw'] * 7 +
    np.random.normal(0, 1.5, len(cdc))
).clip(0, 10).round(1)

# Diet quality from Fruits + Veggies
cdc['diet_quality_score'] = (
    (cdc['Fruits'] + cdc['Veggies']) * 3 +
    np.random.normal(0, 1, len(cdc))
).clip(0, 10).round(1)

# Stress from mental health days
cdc['stress_score'] = (
    cdc['mental_health_days'] / 3 +
    np.random.normal(0, 1, len(cdc))
).clip(0, 10).round(1)

# Sleep score from physical health
cdc['sleep_score'] = (
    8 - (cdc['physical_health_days'] / 6) +
    np.random.normal(0, 1, len(cdc))
).clip(0, 10).round(1)

# Adherence rate
cdc['adherence_rate'] = (
    0.85 - (cdc['diabetes_label'] * 0.1) +
    np.random.normal(0, 0.1, len(cdc))
).clip(0.1, 1.0).round(2)

cdc['missed_doses'] = np.round(
    (1 - cdc['adherence_rate']) * 21
).astype(int).clip(0, 15)

cdc['has_chronic_condition'] = (
    (cdc['diabetes_label'] > 0) |
    (cdc['HighBP'] == 1) |
    (cdc['Stroke'] == 1)
).astype(int)

cdc['allergies_count'] = np.random.choice(
    [0,1,2,3], len(cdc), p=[0.5,0.25,0.15,0.10]
)

# Assign diabetes risk
def assign_diabetes_risk_cdc(row):
    if row['diabetes_label'] == 2 or row['avg_glucose'] > 140:
        return 'high'
    elif row['diabetes_label'] == 1 or row['avg_glucose'] > 100:
        return 'medium'
    else:
        return 'low'

cdc['diabetes_risk'] = cdc.apply(assign_diabetes_risk_cdc, axis=1)

print(f"✅ CDC cleaned: {cdc.shape}")
print(f"\nDiabetes Risk Distribution:")
print(cdc['diabetes_risk'].value_counts())

✅ CDC cleaned: (253680, 38)

Diabetes Risk Distribution:
diabetes_risk
medium    142161
low        68009
high       43510
Name: count, dtype: int64


In [8]:
diabetes_cols = [
    'age', 'bmi', 'avg_glucose', 'sugar_variability',
    'avg_systolic', 'avg_diastolic', 'bp_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'allergies_count',
    'physical_activity_score', 'diet_quality_score',
    'stress_score', 'sleep_score', 'diabetes_risk'
]

synthetic_diabetes = synthetic[diabetes_cols].copy()
print(f"✅ Synthetic diabetes subset: {synthetic_diabetes.shape}")

✅ Synthetic diabetes subset: (50000, 17)


In [9]:
# Select same columns from pima and cdc
pima_final = pima[diabetes_cols].copy()
cdc_final = cdc[diabetes_cols].copy()

# Combine
combined = pd.concat([
    pima_final,
    cdc_final,
    synthetic_diabetes
], ignore_index=True)

# Shuffle
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Fill any nulls
combined = combined.fillna(combined.median(numeric_only=True))

print(f"✅ Combined dataset: {combined.shape}")
print(f"\nDiabetes Risk Distribution:")
print(combined['diabetes_risk'].value_counts())
print(f"\nNull values: {combined.isnull().sum().sum()}")

✅ Combined dataset: (304448, 17)

Diabetes Risk Distribution:
diabetes_risk
medium    158757
low        81713
high       63978
Name: count, dtype: int64

Null values: 0


In [10]:
print("📊 Final Diabetes Dataset Summary:")
print(f"   Total rows: {len(combined):,}")
print(f"   Total columns: {len(combined.columns)}")
print(f"   Columns: {list(combined.columns)}")

print(f"\nFeature Statistics:")
print(combined.describe().round(2))

# Save
output_path = '../data/processed/diabetes_features.csv'
combined.to_csv(output_path, index=False)

print(f"\n✅ Diabetes features saved!")
print(f"   Path: {output_path}")
print(f"   Rows: {len(combined):,}")
print(f"   Size: {os.path.getsize(output_path)/1024/1024:.1f} MB")

📊 Final Diabetes Dataset Summary:
   Total rows: 304,448
   Total columns: 17
   Columns: ['age', 'bmi', 'avg_glucose', 'sugar_variability', 'avg_systolic', 'avg_diastolic', 'bp_variability', 'avg_pulse', 'adherence_rate', 'missed_doses', 'has_chronic_condition', 'allergies_count', 'physical_activity_score', 'diet_quality_score', 'stress_score', 'sleep_score', 'diabetes_risk']

Feature Statistics:
             age        bmi  avg_glucose  sugar_variability  avg_systolic  \
count  304448.00  304448.00    304448.00          304448.00     304448.00   
mean       55.04      27.82       120.58               8.83        130.44   
std        15.89       6.63        32.91               6.87         15.87   
min        18.00      10.40        44.00               0.00         85.00   
25%        42.00      24.00        99.70               3.90        117.60   
50%        57.00      27.00       112.10               6.70        128.20   
75%        67.00      31.00       128.40              12.60 